# Exploring a MODFLOW 6 DISU grid with pyramids-mesh

**MODFLOW 6 DISU** (*discretisation by unstructured cells*) is the most general MODFLOW 6 grid. Instead of
rows and columns (DIS) or a vertex tessellation (DISV), every cell and every connection is listed explicitly
in an IA/JA adjacency table, so the cells need not tile the plane regularly at all. That makes DISU the right
tool for nested grids, cross-sections and — as here — **radial** models that no regular grid can represent.

This notebook reads the USGS example **`ex-gwf-rad-disu`**, a small radial (concentric-ring) aquifer model,
with the `modflow6_disu` driver. The driver parses the ASCII `.disu` file into a
`pyramids.netcdf.ugrid.MeshDataset` — an ordinary 2-D mesh — and from there every pyramids exploration and GIS
operation applies unchanged. The model is deliberately tiny (45 nodes, 550 cells), which makes the raw
topology easy to read.

## Imports

One import line is the whole setup: `MeshDataset` is the object every `pyramids-mesh` driver hands back.
Importing from `pyramids` also puts its vendored GDAL/PROJ data in place, so nothing here is order-sensitive
in a notebook — just import the names you use.

In [ ]:
from pyramids.netcdf.ugrid import MeshDataset
import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

## Locate the example data

This notebook lives in `docs/examples/real-data/`, so the repo-root `examples/data/` folder is three levels
up. Paths inside a notebook are written **relative to the notebook itself**. The `.disu` file sits in its
extracted example directory alongside the other MODFLOW input files (`.nam`, `.npf`, `.ic`, …); we leave those
in place and point `read_file` at the `.disu`. The one rule for `read_file` is to pass a `pathlib.Path`, never
`str(...)`.

In [ ]:
from pathlib import Path
DATA = Path("../../../examples/data/real-data")
mesh_file = DATA / "modflow6" / "ex-gwf-rad-disu" / "ex-gwf-rad-disu.disu"
mesh_file, mesh_file.exists()

## Read the file

`MeshDataset.read_file(path, driver=...)` parses the file and returns a `MeshDataset` — the same type pyramids
produces from a NetCDF UGRID file. We name the `"modflow6_disu"` driver explicitly and pass the **`Path`**
object. (The read may warn that some faces were re-wound counter-clockwise; that is the normaliser fixing
winding order, not an error.)

In [ ]:
ds = MeshDataset.read_file(mesh_file, driver="modflow6_disu")
type(ds).__name__

### Let cleopatra pick the plotting backend

cleopatra switches matplotlib to a headless backend on the first read. Calling its chooser once, right after
that first read, selects the inline backend in a notebook (and falls back to `Agg` in a plain script or CI),
so every later `plot_outline()` / `plot(...)` embeds in the cell output.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## Explore the dataset

A `MeshDataset` exposes summary properties: `n_node` / `n_face` (mesh size), `crs` / `epsg` (georeferencing,
if the format recorded any), `bounds` (the `(minx, miny, maxx, maxy)` extent) and `data_variable_names`
(attached field variables).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(float(b), 3) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

Just **45 nodes** carry **550 cells** — far more cells than nodes, the hallmark of a radial fan where
many wedge cells share the same handful of ring-boundary vertices. `crs`/`epsg` are `None`: DISU uses raw
model coordinates. The extent is the giveaway for *radial* — x runs `0 → 2000` (the **radius**) while y runs
`0 → 12566 ≈ 2π·2000` (the **arc length** of the outer ring). The grid is stored *unrolled* into
(radius, arc-length) coordinates rather than as literal circles. `data_variable_names` is empty — topology
only.

## The cell polygons: connectivity shape

`ds.mesh.face_node_connectivity.data` is the `(n_face, k)` table of node indices per cell, where `k` is the
widest polygon. Cells with fewer nodes than `k` are **padded with `-1` (fill)**. Counting the non-negative
entries per row gives each cell's node count — its *degree*.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
degree = (conn >= 0).sum(axis=1)
vals, counts = np.unique(degree, return_counts=True)
dist = pd.Series(dict(zip(vals.tolist(), counts.tolist())), name="n_faces")
dist.index.name = "nodes_per_face"
dist

Two cell shapes: **525 quadrilaterals and 25 triangles**, so the `(550, 4)` table carries exactly 25
`-1` fill slots (one per triangle). This mix is pure DISU — a regular grid could not hold it. The 25 triangles
are the innermost **wedge** cells of the radial fan: at radius ≈ 0 a ring quad degenerates to a triangle as
its inner edge collapses onto the centre axis. Everything further out is a quad ring cell.

## The underlying geometry

`ds.mesh` is the `Mesh2d` holding the raw arrays: `node_x` / `node_y` (coordinates), the connectivity table,
`face_areas` (one area per cell) and `face_centroids` (shape `(2, n_face)`). Everything geometric is derived
from the connectivity plus the node coordinates.

In [ ]:
mesh = ds.mesh
areas = mesh.face_areas
pd.Series({
    "n_node": int(mesh.node_x.shape[0]),
    "conn_shape": conn.shape,
    "radius_range_x": (float(mesh.node_x.min()), float(mesh.node_x.max())),
    "arc_range_y": (round(float(mesh.node_y.min()), 1), round(float(mesh.node_y.max()), 1)),
    "area_min": round(float(areas.min()), 3),
    "area_max": round(float(areas.max()), 1),
})

The areas span five orders of magnitude — from **0.2** at the centre to **5.5 million** at the outer
ring. That is the radial signature: a fixed angular sector sweeps out far more area at large radius than near
the axis, so ring cells grow steadily outward. The tiny areas belong to the inner triangles; the largest to
the outermost quads.

## Wireframe: `plot_outline()`

The simplest view needs no data — just the mesh edges. `plot_outline()` draws the wireframe through cleopatra
and embeds it inline; it is the fastest check that a file read in correctly, and here it exposes the radial
structure directly.

In [ ]:
_g = ds.plot_outline()
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

A **fan**. Because the grid is drawn in (radius, arc-length) coordinates, the whole model converges to a
single point on the left (radius 0, the centre axis) and widens to the full outer circumference on the right
(radius 2000). The vertical lines are **concentric ring boundaries** (constant radius); the two sloping edges
are the outer boundary fanning open — a sector's arc length grows in proportion to its radius. Roll this fan
back into polar coordinates and it is a set of nested circles.

## A data map: attach and colour a field

To draw a *filled* map we need a value per cell. These files carry geometry only, so we synthesise one — each
cell's area — and attach it with `with_variable(..., location="face")`, which returns a **new** dataset (the
original `ds` is untouched). Then `plot` colours the mesh by it.

In [ ]:
d2 = ds.with_variable("cell_area", ds.mesh.face_areas, location="face")
_g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area")
if ds.epsg:
    add_tiles(_g.ax, crs=ds.epsg)

The colour ramp runs straight up the radius: near-zero area at the apex (centre) brightening to the
maximum at the outer ring. A real model's head or velocity field would plot exactly the same way — this is the
generic pyramids data-map path, indifferent to what the per-cell values mean.

### Statistical summary

With a field attached, `.summary()` reports per-variable statistics (count, min, max, mean, std).

In [ ]:
d2.summary()

## GIS operations, for free

Because the mesh is a first-class pyramids object, the pyramids GIS layer applies directly — `pyramids-mesh`
itself adds none of this. Each cell below runs one operation and interprets it.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every cell into a polygon row of a geopandas `GeoDataFrame` — the standard vector
structure for export to shapefile/GeoPackage or for spatial joins.

In [ ]:
gdf = ds.to_geodataframe()
gdf.head()

550 polygon rows, geometry only. The first rows are small triangles near the centre axis (three distinct
corners before the ring closes); further out they become the four-corner ring quads — the exact shape mix the
degree table predicted.

### Cell adjacency and connected components

`sparse_adjacency()` is the cell-to-cell adjacency as a SciPy sparse matrix (`nnz` = twice the number of
shared internal edges). `connected_components()` labels which cells are mutually reachable through that
adjacency.

In [ ]:
adj = ds.sparse_adjacency()
labels = ds.connected_components()
pd.Series({
    "adjacency_shape": adj.shape,
    "nnz": int(adj.nnz),
    "n_components": int(np.unique(labels).size),
})

The adjacency is very sparse (`nnz` is tiny for 550 cells) and the grid splits into **hundreds** of
components rather than one. A fully unstructured model wires its cells through the solver's IA/JA table, not
through coincident polygon edges, so the geometry-based edge tracer finds few shared edges and most cells read
as isolated. That is a genuine trait of DISU grids — the hydraulic connectivity lives in the model files, not
in the cell outlines — not a defect in the read.

## Takeaway

- **DISU = fully unstructured**: cells and connections listed explicitly, read straight into a `MeshDataset`
  with no georeferencing (raw model coordinates).
- `ex-gwf-rad-disu` is a **radial** grid stored *unrolled* into (radius, arc-length) coordinates — hence the
  fan-shaped wireframe and the cell areas that climb from 0.2 at the centre to 5.5 million at the rim.
- The cell mix is real DISU: **525 quads + 25 inner-wedge triangles**, encoded with 25 `-1` fill slots in the
  `(550, 4)` connectivity table.
- Once read, it is an ordinary pyramids mesh: synthesise a field and `plot` it, or drop into the GIS layer
  (`to_geodataframe`, `sparse_adjacency`, `connected_components`) for free. The sparse edge adjacency and many
  components are the expected signature of an unstructured grid, whose true connectivity lives in the model's
  IA/JA table.